In [1]:
import pandas as pd
import json
import ast
from pathlib import Path
from typing import List, Dict, Any
import re

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
def extract_regulatory_constraints(csv_file_path: str, output_file_path: str = None) -> pd.DataFrame:
    """
    Extract regulatory constraints from LLM output CSV and organize into structured format.
    
    Args:
        csv_file_path: Path to the CSV file containing LLM outputs
        output_file_path: Optional path to save the processed CSV
        
    Returns:
        DataFrame with regulatory constraints organized by rows
    """
    # Read the original CSV
    df = pd.read_csv(csv_file_path)
    
    # Filter for successfully parsed entries only
    df_parsed = df[df['parsed_successfully'] == True].copy()
    
    print(f"Processing {len(df_parsed)} successfully parsed entries out of {len(df)} total entries")
    
    # List to store extracted constraint data
    constraints_data = []
    
    for idx, row in df_parsed.iterrows():
        try:
            # Parse the JSON from raw_output
            raw_output = row['raw_output']
            
            # Handle different JSON formats (with or without markdown code blocks)
            if '```json' in raw_output:
                json_str = raw_output.split('```json')[1].split('```')[0].strip()
            else:
                json_str = raw_output.strip()
            
            # Parse JSON
            parsed_data = json.loads(json_str)
            
            # Extract document metadata
            doc_metadata = parsed_data.get('document_metadata', {})
            regulatory_constraints = parsed_data.get('regulatory_constraints', [])
            
            # Skip if no regulatory constraints
            if not regulatory_constraints:
                continue
                
            # Process each constraint
            for constraint in regulatory_constraints:
                constraint_row = {
                    # Document metadata columns
                    'model_name': row['model_name'],
                    'document_number': doc_metadata.get('document_number'),
                    'type_of_wind_farm': doc_metadata.get('type_of_wind_farm'),
                    'chunk_number': row['chunk_number'],
                    'document_id': row['document_id'],
                    
                    # Regulatory constraint columns
                    'constraint_type': constraint.get('type'),
                    'requirement': constraint.get('requirement'),
                    'scope': constraint.get('scope'),
                    'numerical_value': constraint.get('numerical_value'),
                    'unit': constraint.get('unit'),
                    'source': constraint.get('source'),
                    'related_domains': _format_related_domains(constraint.get('related_domains'))
                }
                
                constraints_data.append(constraint_row)
                
        except (json.JSONDecodeError, KeyError, AttributeError) as e:
            print(f"Error processing row {idx}: {e}")
            continue
    
    # Create DataFrame
    constraints_df = pd.DataFrame(constraints_data)
    
    print(f"Extracted {len(constraints_df)} regulatory constraints")
    
    # Save to file if output path provided
    if output_file_path:
        constraints_df.to_csv(output_file_path, index=False)
        print(f"Results saved to {output_file_path}")
    
    return constraints_df

def _format_related_domains(domains) -> str:
    """Format related domains list into a readable string."""
    if not domains:
        return None
    if isinstance(domains, list):
        return '; '.join(domains)
    return str(domains)

print("Function defined successfully!")

Function defined successfully!


In [3]:
def create_summary_statistics(constraints_df: pd.DataFrame) -> Dict[str, Any]:
    """
    Create summary statistics for the extracted regulatory constraints.
    
    Args:
        constraints_df: DataFrame with extracted constraints
        
    Returns:
        Dictionary with summary statistics
    """
    stats = {
        'total_constraints': len(constraints_df),
        'unique_documents': constraints_df['document_id'].nunique(),
        'constraints_by_type': constraints_df['constraint_type'].value_counts().to_dict(),
        'constraints_by_model': constraints_df['model_name'].value_counts().to_dict(),
        'constraints_with_numerical_values': constraints_df['numerical_value'].notna().sum(),
        'most_common_sources': constraints_df['source'].value_counts().head(10).to_dict(),
        'wind_farm_types': constraints_df['type_of_wind_farm'].value_counts().to_dict()
    }
    
    return stats

def print_summary_statistics(stats: Dict[str, Any]):
    """Print formatted summary statistics."""
    print("=" * 50)
    print("REGULATORY CONSTRAINTS SUMMARY")
    print("=" * 50)
    print(f"Total Constraints Extracted: {stats['total_constraints']}")
    print(f"Unique Documents: {stats['unique_documents']}")
    print(f"Constraints with Numerical Values: {stats['constraints_with_numerical_values']}")
    
    print("\nConstraints by Type:")
    for constraint_type, count in stats['constraints_by_type'].items():
        print(f"  {constraint_type}: {count}")
    
    print("\nConstraints by Model:")
    for model, count in stats['constraints_by_model'].items():
        print(f"  {model}: {count}")
    
    print("\nWind Farm Types:")
    for farm_type, count in stats['wind_farm_types'].items():
        print(f"  {farm_type}: {count}")
    
    print("\nMost Common Sources:")
    for source, count in list(stats['most_common_sources'].items())[:5]:
        source_name = source if source else "Not specified"
        print(f"  {source_name}: {count}")

print("Summary functions defined successfully!")

Summary functions defined successfully!


In [4]:
def process_multiple_csv_files(input_directory: str, output_directory: str = None) -> pd.DataFrame:
    """
    Process multiple CSV files from a directory and combine results.
    
    Args:
        input_directory: Directory containing CSV files from LLM processing
        output_directory: Directory to save processed results
        
    Returns:
        Combined DataFrame with all regulatory constraints
    """
    input_path = Path(input_directory)
    
    if output_directory:
        output_path = Path(output_directory)
        output_path.mkdir(exist_ok=True)
    else:
        output_path = input_path / "processed"
        output_path.mkdir(exist_ok=True)
    
    # Find all CSV files
    csv_files = list(input_path.glob("*.csv"))
    
    if not csv_files:
        print(f"No CSV files found in {input_directory}")
        return pd.DataFrame()
    
    print(f"Found {len(csv_files)} CSV files to process")
    
    all_constraints = []
    
    for csv_file in csv_files:
        print(f"\nProcessing {csv_file.name}...")
        
        try:
            # Extract constraints from this file
            constraints_df = extract_regulatory_constraints(str(csv_file))
            
            if len(constraints_df) > 0:
                # Add source file information temporarily for tracking
                constraints_df['source_file'] = csv_file.name
                all_constraints.append(constraints_df)
                
                # Save individual processed file (without source_file column)
                individual_df = constraints_df.drop(columns=['source_file'])
                individual_output = output_path / f"processed_{csv_file.name}"
                individual_df.to_csv(individual_output, index=False)
                print(f"  Saved individual results to {individual_output}")
            else:
                print(f"  No constraints found in {csv_file.name}")
                
        except Exception as e:
            print(f"  Error processing {csv_file.name}: {e}")
    
    # Combine all results
    if all_constraints:
        combined_df = pd.concat(all_constraints, ignore_index=True)
        
        # Remove source_file column from combined results
        final_df = combined_df.drop(columns=['source_file'])
        
        # Save combined results
        combined_output = output_path / "combined_regulatory_constraints.csv"
        final_df.to_csv(combined_output, index=False)
        print(f"\nCombined results saved to {combined_output}")
        
        # Generate and print summary
        stats = create_summary_statistics(final_df)
        print_summary_statistics(stats)
        
        return final_df
    else:
        print("No constraints extracted from any files")
        return pd.DataFrame()

def clean_existing_files(directory_path: str):
    """
    Remove source_file column from existing processed CSV files.
    
    Args:
        directory_path: Path to directory containing processed CSV files
    """
    dir_path = Path(directory_path)
    
    if not dir_path.exists():
        print(f"Directory {directory_path} does not exist")
        return
    
    csv_files = list(dir_path.glob("processed_*.csv")) + list(dir_path.glob("combined_*.csv"))
    
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            
            if 'source_file' in df.columns:
                df_cleaned = df.drop(columns=['source_file'])
                df_cleaned.to_csv(csv_file, index=False)
                print(f"Removed source_file column from {csv_file.name}")
            else:
                print(f"No source_file column found in {csv_file.name}")
                
        except Exception as e:
            print(f"Error processing {csv_file.name}: {e}")

print("Batch processing function defined successfully!")

Batch processing function defined successfully!


# Example Usage

In [5]:
# # Quick test with the provided CSV file
# input_file = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/LLM_Results/gemini_27_results.csv"
# output_file = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Processed_Results/gemini_27_processed.csv"

# print("Processing gemini_27_results.csv...")
# constraints_df = extract_regulatory_constraints(input_file, output_file)

# if len(constraints_df) > 0:
#     stats = create_summary_statistics(constraints_df)
#     print_summary_statistics(stats)
    
#     print("\nSample of extracted constraints:")
#     print(constraints_df[['constraint_type', 'requirement', 'numerical_value', 'unit']].head(10))
# else:
#     print("No constraints found in the file.")

# Process the entire folder

In [6]:
# mac
# input_directory = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/LLM_Results"
# output_directory = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Processed_Results"

# PC
input_directory = "D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\LLM_Results"
output_directory = "D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Processed_Results"

In [7]:
print("Starting batch processing of all CSV files...")
print(f"Input directory: {input_directory}")
print(f"Output directory: {output_directory}")

# Process all CSV files in the directory
combined_constraints_df = process_multiple_csv_files(input_directory, output_directory)

if len(combined_constraints_df) > 0:
    print(f"\n✅ Processing complete! Combined dataset has {len(combined_constraints_df)} constraints.")
    print(f"Data covers {combined_constraints_df['document_id'].nunique()} unique documents.")
    print(f"Models used: {', '.join(combined_constraints_df['model_name'].unique())}")
else:
    print("❌ No constraints were extracted from any files.")

Starting batch processing of all CSV files...
Input directory: D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\LLM_Results
Output directory: D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Processed_Results
Found 302 CSV files to process

Processing claude_10_results.csv...
Processing 16 successfully parsed entries out of 442 total entries
Extracted 71 regulatory constraints
  Saved individual results to D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Processed_Results\processed_claude_10_results.csv

Processing claude_11_results.csv...
Processing 17 successfully parsed entries out of 532 total entries
Extracted 118 regulatory constraints
  Saved individual results to D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Processed_Results\processed_claude_11_results.csv

Processing claude_12_results.csv...
Processing 11 successfully parsed entries out of 161 total entries
Extracted

In [8]:
# # Clean existing files to remove source_file column
# print("Cleaning existing processed files...")
# clean_existing_files(output_directory)

# # Optional: Display a sample of the cleaned results
# if 'combined_constraints_df' in locals() and len(combined_constraints_df) > 0:
#     print("\nSample of cleaned regulatory constraints:")
#     print(combined_constraints_df[['model_name', 'constraint_type', 'requirement', 'numerical_value', 'unit']].head(15))
    
#     # Show breakdown by model
#     print("\nConstraints by model:")
#     print(combined_constraints_df['model_name'].value_counts())